# 02. 실습: 순차 확률 예측과 강화학습 정책 구분하기

## 학습 목표

- MDP의 상태, 행동, 보상, 전이와 종료 조건을 명시한다.
- 전환 확률을 수동적으로 예측하는 모델과 고객 상태에 개입하는 정책을 구분한다.
- 작은 영업 환경에서 Q-learning으로 행동 가치를 학습하고 무작위 정책과 비교한다.
- `action = conversion probability`, `reward = accuracy`라는 서술만으로는 판매 최적화 RL 문제가 완전히 정의되지 않는 이유를 설명한다.

> **Toy reproduction 주의:** 이 노트북의 환경과 결과는 개념 검증용으로 직접 만든 것이다. 논문의 데이터·구현·보고된 96.7% 정확도나 43.2% 전환 향상을 재현하지 않으며, 공개된 후속 모델·데이터 아티팩트도 사용하지 않는다. 외부 API와 네트워크는 필요 없다.

## 1. 두 문제를 수학적으로 분리하기

### 수동적 순차 예측

현재까지의 대화 이력 $h_t$로 최종 전환 $Y$의 확률 $\hat p_t=P(Y=1\mid h_t)$를 추정한다. log-loss나 Brier score는 **예측 확률의 정확성**을 평가하지만, $\hat p_t$ 자체가 고객에게 전달되는 제안은 아니다.

### 개입 정책을 가진 MDP

- 상태 $s_t$: `(관심도, 남은 인내, 할인 사용 여부, 턴)`
- 행동 $a_t$: `경청`, `데모`, `할인 제안`
- 전이 $P(s_{t+1}\mid s_t,a_t)$: 선택한 행동이 다음 관심도와 인내에 영향을 줌
- 보상 $r_t$: 행동 비용과 종료 시 전환 가치의 합
- 정책 $\pi(a\mid s)$: 상태마다 실제로 실행할 행동을 선택

RL이라고 부르려면 최소한 행동이 무엇을 바꾸는지, 어떤 보상을 언제 받는지, 관측되지 않은 교란과 오프라인 로그의 행동 정책을 어떻게 처리하는지가 정의되어야 한다.

In [ ]:
import numpy as np

SEED = 250323303
ACTION_NAMES = np.array(["경청", "데모", "할인"])
N_ACTIONS = len(ACTION_NAMES)
HORIZON = 5

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30.0, 30.0)))

class TinySalesEnv:
    """작고 완전히 명시된 교육용 영업 MDP."""
    def __init__(self, rng):
        self.rng = rng

    def reset(self):
        self.interest = int(self.rng.integers(0, 3))
        self.patience = 3
        self.discount_used = 0
        self.turn = 0
        return self.state

    @property
    def state(self):
        return (self.interest, self.patience, self.discount_used, self.turn)

    def step(self, action):
        assert 0 <= action < N_ACTIONS
        # 행동마다 관심도가 오를 조건과 즉시 비용이 다르다.
        if action == 0:       # 경청: 초기에도 비교적 안전하다.
            up_probability, cost = 0.50 + 0.05 * (self.interest < 2), 0.01
        elif action == 1:     # 데모: 이미 관심이 있을 때 효과적이다.
            up_probability, cost = (0.72 if self.interest >= 2 else 0.23), 0.025
        else:                 # 할인: 관심을 끌 수 있지만 마진을 낮춘다.
            up_probability, cost = 0.62, 0.15
            self.discount_used = 1
        if self.rng.random() < up_probability:
            self.interest = min(4, self.interest + 1)
        elif self.rng.random() < 0.18:
            self.interest = max(0, self.interest - 1)
        self.patience -= 1
        self.turn += 1
        done = self.turn >= HORIZON or self.patience <= 0 or self.interest >= 4
        reward = -cost
        converted = 0
        if done:
            conversion_probability = sigmoid(-3.1 + 1.25 * self.interest + 0.35 * self.discount_used)
            converted = int(self.rng.random() < conversion_probability)
            reward += converted
        return self.state, reward, done, converted

env = TinySalesEnv(np.random.default_rng(SEED))
print("초기 상태 예시:", env.reset(), "| 행동:", ACTION_NAMES.tolist())

## 2. 수동적 예측기는 정책이 아니다

무작위 행동 로그에서 상태별 전환 빈도를 추정한다. 이 추정치는 위험도 표시나 상담원 보조 신호로는 쓸 수 있지만, 확률 숫자를 출력하는 것만으로 환경의 전이가 바뀌지는 않는다. 아래 데이터는 예측과 개입의 차이를 드러내기 위한 것이다.

In [ ]:
def run_episode(env, policy, learn_callback=None):
    state = env.reset()
    trajectory, total_reward, converted = [], 0.0, 0
    done = False
    while not done:
        action = int(policy(state))
        next_state, reward, done, converted = env.step(action)
        trajectory.append((state, action, reward, next_state, done))
        if learn_callback is not None:
            learn_callback(state, action, reward, next_state, done)
        total_reward += reward
        state = next_state
    return total_reward, converted, trajectory

behavior_rng = np.random.default_rng(SEED + 1)
behavior_env = TinySalesEnv(np.random.default_rng(SEED + 2))
logs = []
for _ in range(6000):
    _, outcome, trajectory = run_episode(behavior_env, lambda state: behavior_rng.integers(N_ACTIONS))
    for state, action, reward, next_state, done in trajectory:
        logs.append((state, outcome))

# 각 상태에서 관찰된 최종 전환율: 행동을 선택하는 규칙이 아니라 조건부 빈도 추정이다.
state_sum, state_count = {}, {}
for state, outcome in logs:
    state_sum[state] = state_sum.get(state, 0) + outcome
    state_count[state] = state_count.get(state, 0) + 1
forecast = {state: (state_sum[state] + 1) / (state_count[state] + 2) for state in state_count}
sample_states = sorted(forecast, key=lambda state: (state[3], state[0]))[:8]
for state in sample_states:
    print(f"state={state}, 예측 전환확률={forecast[state]:.3f}, 표본수={state_count[state]}")

assert all(0.0 < probability < 1.0 for probability in forecast.values())
print("예측기는 상태에 점수를 붙였지만 실행할 행동을 아직 정의하지 않았다.")

## 3. Q-learning으로 개입 정책 학습

Q-learning은 $Q(s,a) \leftarrow Q(s,a)+\alpha[r+\gamma\max_{a'}Q(s',a')-Q(s,a)]$로 상태-행동 가치를 갱신한다. 여기서는 표가 작으므로 딕셔너리를 쓰고, 탐색률을 점차 줄인다. 평가 때는 별도의 고정 seed 환경을 사용한다.

In [ ]:
train_rng = np.random.default_rng(SEED + 3)
train_env = TinySalesEnv(np.random.default_rng(SEED + 4))
Q = {}

def q_values(state):
    if state not in Q:
        Q[state] = np.zeros(N_ACTIONS, dtype=float)
    return Q[state]

for episode in range(30000):
    epsilon = max(0.04, 0.9 * (1.0 - episode / 30000))
    def exploratory_policy(state):
        if train_rng.random() < epsilon:
            return int(train_rng.integers(N_ACTIONS))
        return int(np.argmax(q_values(state)))
    def update(state, action, reward, next_state, done):
        target = reward if done else reward + 0.96 * np.max(q_values(next_state))
        q_values(state)[action] += 0.12 * (target - q_values(state)[action])
    run_episode(train_env, exploratory_policy, update)

def evaluate(policy, seed, episodes=8000):
    eval_env = TinySalesEnv(np.random.default_rng(seed))
    rewards, conversions = [], []
    for _ in range(episodes):
        reward, converted, _ = run_episode(eval_env, policy)
        rewards.append(reward)
        conversions.append(converted)
    rewards = np.asarray(rewards)
    conversions = np.asarray(conversions)
    return rewards.mean(), conversions.mean(), rewards.std(ddof=1) / np.sqrt(len(rewards))

random_policy_rng = np.random.default_rng(SEED + 5)
random_result = evaluate(lambda state: random_policy_rng.integers(N_ACTIONS), SEED + 6)
learned_result = evaluate(lambda state: np.argmax(q_values(state)), SEED + 6)
print(f"무작위 정책: 평균 보상={random_result[0]:.3f}, 전환율={random_result[1]:.3f}, SE={random_result[2]:.4f}")
print(f"학습 정책  : 평균 보상={learned_result[0]:.3f}, 전환율={learned_result[1]:.3f}, SE={learned_result[2]:.4f}")
for state in [(0, 3, 0, 0), (2, 3, 0, 0), (3, 2, 0, 1), (1, 2, 1, 1)]:
    print(f"state={state} -> {ACTION_NAMES[np.argmax(q_values(state))]} | Q={np.round(q_values(state), 3)}")

assert all(np.all(np.isfinite(values)) for values in Q.values())
assert learned_result[0] > random_result[0] + 0.02, "학습 정책이 이 toy 환경의 무작위 기준선보다 좋아야 한다."
print("검증 통과: 실제 행동이 전이를 바꾸는 MDP에서 정책 가치가 개선되었다.")

## 해석과 논문 읽기 체크리스트

이 toy 환경에서는 행동이 고객 상태와 비용을 바꾸므로 RL 목적이 명확하다. 반면 **확률을 action으로, 정답 정확도를 reward로 둔다**는 설명은 보통 다음 중 무엇인지 추가 명세가 필요하다.

1. 확률 예측을 proper scoring rule로 학습하는 순차 지도학습인가?
2. 임계값 선택·자원 배분처럼 확률 출력이 실제 업무 행동으로 연결되는가?
3. 상담 전략을 선택하고 고객 반응이라는 전이를 관측하는 정책 학습인가?
4. 오프라인 로그라면 행동 확률(propensity), 지원집합, 반사실 평가를 어떻게 처리했는가?

accuracy 보상은 0.49와 0.01을 모두 같은 ‘비전환 정답’으로 취급할 수 있어 확률 보정에 부적합하다. 확률 예측에는 log-loss/Brier 같은 proper scoring rule을, 사업 개입에는 수익·비용·고객 피해를 반영한 정책 보상과 무작위 또는 타당한 인과 평가를 별도로 설계해야 한다.